## Lakehouse Discovery
Lists all lakehouses available in this workspace with their ABFS paths.
Run this to confirm the correct lakehouses are connected before executing the notebook.

In [1]:
import notebookutils

lakehouses = notebookutils.lakehouse.list()
for lh in lakehouses:
    print(f'Name: {lh.displayName}')
    print(f'ABFS: abfss://{lh.workspaceId}@onelake.dfs.fabric.microsoft.com/{lh.id}.Lakehouse/Tables')
    print('---')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 5, Finished, Available, Finished, False)

Name: StagingLakehouseForDataflows_20260514123859
ABFS: abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/faaaf149-9b0f-415d-b366-0e86de95a7e3.Lakehouse/Tables
---
Name: CRM_Monitoring
ABFS: abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/4eaaee48-ced4-4efc-9115-c094bf6646f6.Lakehouse/Tables
---
Name: dataverse_esacontactde_cds2_workspace_org40d0fe68
ABFS: abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/6cb27148-a409-4e8d-b5f8-453c40451ec5.Lakehouse/Tables
---
Name: dataverse_esacontact_cds2_workspace_a94a4bf848e144bba608bb2eb51cbe
ABFS: abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/b0c4cd42-939f-49cc-a936-34774e4eff33.Lakehouse/Tables
---


## Configuration
Update these values if credentials or paths change.
- TENANT_ID / CLIENT_ID / CLIENT_SECRET — Azure app registration for Dataverse API access
- DATAVERSE_URL — the CRM environment endpoint
- ENVIRONMENT_ID — Dataverse environment identifier
- LAKEHOUSE_PATH — ABFS path for the CRM_Monitoring lakehouse (get from Fabric > Lakehouse > Properties)

In [2]:
# ============================================================
# CONFIGURATION — update these if credentials change
# ============================================================
TENANT_ID     = '9a5cacd0-2bef-4dd7-ac5c-7ebe1f54f495'
CLIENT_ID     = '28d48667-10ad-4563-93c3-499438dafbab'
CLIENT_SECRET = '<SET_VIA_SECURE_CONFIGURATION>'
DATAVERSE_URL  = 'https://esacontact.crm4.dynamics.com'
ENVIRONMENT_ID = '2c31dc4f-f142-4c34-b691-e24228103ea2'

# Your Lakehouse ABFS path
# Get this from Fabric → your Lakehouse → Properties → ABFS path
LAKEHOUSE_PATH = 'abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/b0c4cd42-939f-49cc-a936-34774e4eff33.Lakehouse/Tables'

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 6, Finished, Available, Finished, False)

## Dataverse Authentication
Acquires a bearer token from Microsoft identity platform using the app credentials above.
Sets the request headers and snapshot timestamp used by all subsequent API calls.
If this fails, check the CLIENT_SECRET hasn't expired in Azure portal.

In [3]:
import msal
import requests
import pandas as pd
from datetime import datetime, timezone

def get_dataverse_token():
    app = msal.ConfidentialClientApplication(
        CLIENT_ID,
        authority=f'https://login.microsoftonline.com/{TENANT_ID}',
        client_credential=CLIENT_SECRET
    )
    result = app.acquire_token_for_client(
        scopes=[f'{DATAVERSE_URL}/.default']
    )
    if 'access_token' in result:
        print('✅ Dataverse token acquired')
        return result['access_token']
    else:
        raise Exception(f'Auth failed: {result.get("error_description")}')

token = get_dataverse_token()
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json',
    'Prefer': 'odata.include-annotations="*"'
}
snapshot_time = datetime.now(timezone.utc)
print(f'Snapshot time: {snapshot_time}')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 7, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/msal/application.py:219: UserWarning: Please upgrade msal-extensions. Only msal-extensions 1.2+ can work with msal 1.30+
  warnings.warn(


## API Call 1: Organisation Resources
Fetches the current Dataverse environment stats including active users, 
custom entities, and published workflows. Used to populate the health snapshot.

In [4]:
# ---- Call 1: Organisation Resources ----
url = f'{DATAVERSE_URL}/api/data/v9.2/RetrieveOrganizationResources'
response = requests.get(url, headers=headers)
response.raise_for_status()
org = response.json().get('OrganizationResources', {})
print('✅ Org Resources fetched')
print(org)

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 8, Finished, Available, Finished, False)

✅ Org Resources fetched
{'CurrentNumberOfActiveUsers': 11830, 'MaxNumberOfActiveUsers': 200000, 'CurrentNumberOfNonInteractiveUsers': 0, 'MaxNumberOfNonInteractiveUsers': 7, 'CurrentNumberOfCustomEntities': 79, 'MaxNumberOfCustomEntities': 3000, 'CurrentNumberOfPublishedWorkflows': 0, 'MaxNumberOfPublishedWorkflows': 0, 'CurrentStorage': 0, 'MaxStorage': 511488000}


## API Call 2: Failed System Jobs
Queries async operations for failed jobs (statecode 3, statuscode 31).
A high failed_jobs count can indicate sync or plugin issues in the environment.

In [5]:
# ---- Call 2: Failed system jobs count ----
url = f'{DATAVERSE_URL}/api/data/v9.2/asyncoperations'
params = {
    '$filter': 'statecode eq 3 and statuscode eq 31',
    '$count': 'true',
    '$top': '1',
    '$select': 'asyncoperationid'
}
response = requests.get(url, headers=headers, params=params)
response.raise_for_status()
failed_jobs = response.json().get('@odata.count', 0)
print(f'✅ Failed system jobs: {failed_jobs}')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 9, Finished, Available, Finished, False)

✅ Failed system jobs: 387


## API Call 3: Enabled System Users
Counts all active (non-disabled) system users in the Dataverse environment.
This gives the enabled_users value for the health snapshot.

In [6]:
# ---- Call 3: Enabled system users count ----
url = f'{DATAVERSE_URL}/api/data/v9.2/systemusers'
params = {
    '$filter': 'isdisabled eq false',
    '$count': 'true',
    '$top': '1',
    '$select': 'systemuserid'
}
response = requests.get(url, headers=headers, params=params)
response.raise_for_status()
enabled_users = response.json().get('@odata.count', 0)
print(f'✅ Enabled users: {enabled_users}')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 10, Finished, Available, Finished, False)

✅ Enabled users: 5000


## API Call 4: Plugin Statistics
Retrieves plugin execution and crash counts from Dataverse.
High crash percentages here can indicate broken customisations or failing integrations.

In [7]:
# ---- Call 4: Plugin statistics ----
url = f'{DATAVERSE_URL}/api/data/v9.2/plugintypestatistics'
params = {
    '$select': 'plugintypeid,crashcount,crashpercent,executecount,failurepercent'
}
response = requests.get(url, headers=headers, params=params)
response.raise_for_status()
plugins = response.json().get('value', [])
total_executions = sum(p.get('executecount', 0) or 0 for p in plugins)
total_crashes = sum(p.get('crashcount', 0) or 0 for p in plugins)
print(f'✅ Plugin executions: {total_executions}, crashes: {total_crashes}')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 11, Finished, Available, Finished, False)

✅ Plugin executions: 0, crashes: 0


## Build Health Snapshot Row
Combines all API call results into a single row dataframe.
Calculates percentages (active_users_pct, custom_entities_pct) and converts 
storage to GB. Storage pending admin permissions fields are placeholders for now.
Environment is hardcoded to PROD — update

In [8]:
# ---- Combine all data into one row ----
row = {
    'snapshot_date':                    snapshot_time,
    'active_users_current':             org.get('CurrentNumberOfActiveUsers', 0),
    'active_users_max':                 org.get('MaxNumberOfActiveUsers', 0),
    'active_users_pct':                 round(
                                            org.get('CurrentNumberOfActiveUsers', 0) /
                                            max(org.get('MaxNumberOfActiveUsers', 1), 1) * 100, 2
                                        ),
    'non_interactive_users_current':    org.get('CurrentNumberOfNonInteractiveUsers', 0),
    'non_interactive_users_max':        org.get('MaxNumberOfNonInteractiveUsers', 0),
    'custom_entities_current':          org.get('CurrentNumberOfCustomEntities', 0),
    'custom_entities_max':              org.get('MaxNumberOfCustomEntities', 0),
    'custom_entities_pct':              round(
                                            org.get('CurrentNumberOfCustomEntities', 0) /
                                            max(org.get('MaxNumberOfCustomEntities', 1), 1) * 100, 2
                                        ),
    'published_workflows_current':      org.get('CurrentNumberOfPublishedWorkflows', 0),
    'published_workflows_max':          org.get('MaxNumberOfPublishedWorkflows', 0),
    'max_storage_bytes':                org.get('MaxStorage', 0),
    'max_storage_gb':                   round(org.get('MaxStorage', 0) / 1024 / 1024 / 1024, 2),
    'current_storage_bytes':            org.get('CurrentStorage', 0),
    'failed_system_jobs':               failed_jobs,
    'enabled_users':                    enabled_users,
    'plugin_total_executions':          total_executions,
    'plugin_total_crashes':             total_crashes,
    # Storage pending admin permissions
    'db_storage_used_mb':               None,
    'file_storage_used_mb':             None,
    'log_storage_used_mb':              None,
    'storage_pending_permissions':      True,
    'environment': 'PROD'
}

df = pd.DataFrame([row])
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], utc=True)
print('✅ DataFrame built')
display(df)

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 12, Finished, Available, Finished, False)

✅ DataFrame built


SynapseWidget(Synapse.DataFrame, c1be564f-ba91-439e-b1d0-156dd5c7c80a)

## Write to CRM Health Snapshot
Appends the new row to the crm_health_snapshot Delta table in CRM_Monitoring lakehouse.
Uses append mode so historical snapshots are preserved — do not change to overwrite.

In [9]:
spark_df = spark.createDataFrame(df)

spark_df.write \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .mode('append') \
    .saveAsTable('crm_health_snapshot')

print(f'✅ Written to crm_health_snapshot')
print(f'   Rows written: {df.shape[0]}')
print(f'   Snapshot time: {snapshot_time}')

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 13, Finished, Available, Finished, False)

✅ Written to crm_health_snapshot
   Rows written: 1
   Snapshot time: 2026-06-10 16:52:03.862450+00:00


## Verify Write
Confirms the row was written successfully and displays the latest 6 snapshots 
ordered by environment and snapshot date. Use this to sense-check the data 
looks correct before closing the notebook.

In [10]:
count = spark.sql('SELECT COUNT(*) as cnt FROM crm_health_snapshot').collect()[0].cnt
print(f'✅ Table contains {count} total rows')
spark.sql('''
    SELECT 
        environment,
        snapshot_date, 
        active_users_current, 
        active_users_max, 
        active_users_pct,
        custom_entities_current,
        failed_system_jobs, 
        enabled_users 
    FROM crm_health_snapshot 
    ORDER BY environment, snapshot_date DESC
''').show(truncate=False)

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 14, Finished, Available, Finished, False)

✅ Table contains 9 total rows
+-----------+--------------------------+--------------------+----------------+----------------+-----------------------+------------------+-------------+
|environment|snapshot_date             |active_users_current|active_users_max|active_users_pct|custom_entities_current|failed_system_jobs|enabled_users|
+-----------+--------------------------+--------------------+----------------+----------------+-----------------------+------------------+-------------+
|           |2026-06-10 16:52:03.86245 |11830               |200000          |5.92            |79                     |387               |5000         |
|           |2026-06-10 02:00:57.333846|11819               |200000          |5.91            |79                     |87                |5000         |
|           |2026-06-09 02:00:56.546899|11805               |200000          |5.9             |79                     |82                |5000         |
|           |2026-06-05 14:38:24.417474|11747       